<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🛡️ M2 作业 · 给代码审查 Crew 加上 guardrail、hook 和 memory

**一句话**：沿用 M1 作业的 3 个 agent / 3 个 task，加上**结构化输出 + 校验重试 + 读文件 hook + 长期记忆**，让 crew 从「能跑」变成「能信任」。

<span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🏷️ 作业 Homework</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📄 C1M2_Assignment.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧩 前置：M1 作业：自动代码审查 Crew · M2 guardrails / hooks / memory 课</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧪 unittests 自动评分</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">⚙️ config/*.yaml</span>

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**要解决的问题**：M1 的 crew 能出报告，但：
- 安全风险等级可能乱写（`critical`、`severe`…），`highest_risk` 也可能和明细对不上 → 用 **guardrail** 校验，不合格自动让 agent 重做；
- 最终结论可能没有明确动作 → guardrail 强制出现 `approve` / `request changes` / `escalate`；
- 「读 PR 文件」是确定性 I/O，不需要 LLM → 用 **before-kickoff hook** 在开跑前读好；
- 每个 PR 都从零开始 → 开 **memory**，让 agent 记住以前见过的漏洞模式。

</div>

<img src="./images/agents-tasks-diagram.png" width="650">

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 本作业新增的四样东西

</div>

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧱 output_json=PydanticModel</div><div style="font-size:0.92em; line-height:1.6;">让 task 输出符合 schema 的 JSON（存在 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">TaskOutput.json_dict</code>）。<br>下游代码才能精确取 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">json_output[&quot;highest_risk&quot;]</code>，不用猜字段。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🚦 guardrails=[fn]</div><div style="font-size:0.92em; line-height:1.6;">普通函数：收 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">TaskOutput</code>，返回 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">(bool, 值或错误信息)</code>。<br>失败 → 错误信息喂回<b>同一个 agent</b> 重做，最多 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">guardrail_max_retries</code>（默认 3）次，仍失败抛异常。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🪝 before_kickoff_callbacks=[fn]</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">kickoff</code> 一开始、agent 动手前调用：收 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">inputs</code> 字典，返回新字典。<br>这里用来读 PR 文件，塞进 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">inputs[&quot;file_content&quot;]</code>。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #a78bfa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧠 memory=True</div><div style="font-size:0.92em; line-height:1.6;">agent 自动多出 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">Search memory</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">Save to memory</code> 两个工具；记忆存在本机磁盘（LanceDB），<b>跨 kickoff、跨进程</b>都在。</div></div></div></div>

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">一次 kickoff 的数据流</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #a78bfa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">kickoff(inputs={&quot;file_path&quot;: …})</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">read_file_hook</code> 读文件</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">inputs[&quot;file_content&quot;]</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">填进 yaml 里的 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">{file_content}</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #60a5fa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">3 个 task 顺序执行 + guardrail 校验</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #22d3ee; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">Markdown 最终报告</span></div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**评分规则（graded cell）**：只在带 `# GRADED CELL` 的格子里写答案；**不要改变量名 / 函数名**，不要改 `config/` 里的配置，否则 grader 找不到你的答案。
本 notebook 保留了这些格子的 grader metadata，每个练习后面的 `unittests.test_*` 格子都应打印 `All tests passed!`。

</div>

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

📝 **TODO 状态**：8 个练习的 `None` / `START CODE HERE` 都已由你自己填好，本次没有补任何答案，只加了注释。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 1 步：导入模块

</div>

`clean_markdown` 来自同目录 `utils.py`：去掉 LLM 有时包在报告外面的 ```` ```markdown ```` 围栏。

In [1]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool, ScrapeWebsiteTool   # 安全工程师的两个网页工具
from pydantic import BaseModel   # 定义 output_json 的结构
from utils import get_openai_api_key, get_serper_api_key, clean_markdown   # utils.py 用 find_dotenv() 向上找到仓库根目录的 .env
from IPython.display import Markdown, display
import yaml
import os
os.environ["CREWAI_TESTING"] = "true"   # 测试模式：跳过首次运行时「是否开启 tracing」的交互确认

In [2]:
import unittests   # 同目录 unittests.py：每个练习的自动检查
import dill   # 比 pickle 能序列化更多对象，用来保存 result 给 grader

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 2 步：环境变量

</div>

CrewAI 读 `MODEL` 环境变量作为默认 LLM，所以 agent 不传 `llm=` 也会用 `gpt-4o-mini`。

In [3]:
# set up the OpenAI model
os.environ["MODEL"] = "gpt-4o-mini"

# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()

# set the Serper API key for the WebsiteSearchTool
os.environ["SERPER_API_KEY"] = get_serper_api_key()

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 3 步：读 YAML 配置

</div>

agent 的人设、task 的描述都写在 YAML 里，Python 里只补「新功能」参数。下面先读进字典：

In [4]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)   # safe_load：只解析数据，不执行 YAML 里的任意 Python 对象

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

看一眼两份配置文件的原文：

In [5]:
from pathlib import Path

print(Path("config/agents.yaml").read_text())

senior_developer:
  role: >
    Senior Developer
  goal: >
    Evaluate code changes for quality issues and decide which problems must be fixed
  backstory: >
    You're an experienced software engineer with expertise in code quality.
    You analyze code for style issues, potential bugs, and maintainability concerns.
    You can distinguish between critical issues that must be fixed and minor issues
    that are suggestions. You make clear decisions about whether code quality
    problems should block approval or simply be noted as recommendations.
  allow_delegation: false

security_engineer:
  role: >
    Security Engineer
  goal: >
    Identify security vulnerabilities in the pull request and determine their risk level
  backstory: >
    You're a cybersecurity specialist focused on secure coding practices.
    You evaluate code for potential security vulnerabilities and determine
    their risk level. You can identify issues like injection risks, authentication
    problems, and da

In [6]:
print(Path("config/tasks.yaml").read_text())

analyze_code_quality:
  description: >
    1. Review the code changes in the pull request.
    2. Identify any style issues, potential bugs, or maintainability concerns.
    3. For each issue, decide if it's critical (must be fixed) or minor (suggestion).
    PR content: 
    {file_content}
  expected_output: >
    A JSON with keys: 'critical_issues' (array of issues that must be fixed), 
    'minor_issues' (array of suggested improvements), and 'reasoning' (explanation of your decisions).
  name: Analyze Code Quality

review_security:
  description: >
    1. Review the findings from both the code quality and security analyses.
    2. Determine if the pull request can be automatically approved based on these findings.
    3. If approval is not possible, decide whether to request specific changes from the developer or escalate to human review.
    4. Provide a clear explanation of your decision and any required next steps.
    PR content: 
    {file_content}
  expected_output: >
    A J

**YAML 字段 → Python 参数**：`Agent(config=…)` / `Task(config=…)` 会把字典里**与类字段同名的 key** 填进去；同名参数你若在代码里显式传了，**以代码为准**（`process_config` 只填 `None` 的字段）。

| 文件 | key | 对应 | 说明 |
|---|---|---|---|
| `agents.yaml` | `role` / `goal` / `backstory` | `Agent` 必填三件套 | 拼进 system prompt，决定 agent 的「人设」 |
| | `allow_delegation: false` | `Agent.allow_delegation` | 不许把活转给别的 agent |
| | `verbose: true` | `Agent.verbose` | 代码里又传了 `verbose=True`，结果一样 |
| `tasks.yaml` | `description` | `Task.description` | 任务说明；结尾的 `{file_content}` 是**占位符**，kickoff 时用 `inputs["file_content"]` 替换 |
| | `expected_output` | `Task.expected_output` | 告诉 LLM 输出该长什么样（这里写了 JSON 的 key） |
| | `name` | `Task.name` | 日志 / `TaskOutput.name` 里显示的名字 |
| | `context: [analyze_code_quality, …]` | ⚠️ 不能直接用 | 字符串列表只有 `@CrewBase` 项目会解析成 task 对象 |

YAML 语法：`>` 是**折叠块**，多行合成一行（换行变空格），适合写长段 prompt。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**两个配置文件里的坑（已用 crewai 1.15.22 验证）**
- `make_review_decision` 的 `context` 是字符串列表：直接 `Task(config=...)` 会报 `AttributeError: 'str' object has no attribute 'get'`。所以练习 5 里**必须显式传** `context=[analyze_code_quality, review_security]`，代码参数覆盖掉 YAML 里的字符串。
- `review_security` 的 `description` 其实是从决策 task 复制来的（「Review the findings… decide whether to request specific changes…」），并不是在描述安全审查；真正约束它的是 `expected_output` + agent 的 `role/goal`。按评分要求不改配置，知道即可。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 4 步：Agents（与 M1 相同，只是人设来自 YAML）

</div>

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🔎 SerperDevTool</div><div style="font-size:0.92em; line-height:1.6;">工具名 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">Search the internet with Serper</code>：调 Serper（Google 搜索 API），需要 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">SERPER_API_KEY</code>。<br><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">base_url</code>：DeepLearning.AI 课堂里走代理 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">DLAI_SERPER_BASE_URL</code>，本地退回 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">https://google.serper.dev</code>。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">📄 ScrapeWebsiteTool</div><div style="font-size:0.92em; line-height:1.6;">工具名 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">Read website content</code>：抓网页正文。<br>构造时没给 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">website_url</code> → agent 每次调用时自己传 URL（比如搜索结果里的 OWASP 页面）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">👥 三个 agent</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">senior_developer</code>：代码质量<br><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">security_engineer</code>：安全漏洞（唯一有工具的）<br><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">tech_lead</code>：汇总并拍板</div></div></div></div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10 版本差异：`search_url="https://owasp.org"` 不起作用**。crewai_tools 的 `SerperDevTool` 没有 `search_url` 字段，这个参数被 pydantic 静默丢弃（`getattr(tool, "search_url")` 不存在），请求永远发到 `{base_url}/search`，**并不会**把搜索限制在 OWASP 站内。想限定站点，得让 agent 在查询里写 `site:owasp.org …`。按评分要求保留原代码。

</div>

In [7]:
# Create the tool instances for the security engineer agent
serper_search_tool = SerperDevTool(
    search_url="https://owasp.org",
    base_url=os.getenv("DLAI_SERPER_BASE_URL") or "https://google.serper.dev",   # 本地 getenv 得 None → 退回默认地址
)
scrape_website_tool = ScrapeWebsiteTool()   # 不绑定 URL：agent 调用时自己给 website_url


# create the Senior Developer agent
senior_developer = Agent(
    # load role, goal, and backstory from the YAML configuration
    config=agents_config["senior_developer"],
    # set verbose
    verbose=True,
)

# create the Security Engineer agent
security_engineer = Agent(
    # load role, goal, and backstory from the YAML configuration
    config=agents_config["security_engineer"],
    # add the website search tools (you need to unpack the list)
    tools=[serper_search_tool, scrape_website_tool],
    # set verbose
    verbose=True,
)

# create the Tech Lead agent
tech_lead = Agent(
    # load role, goal, and backstory from the YAML configuration
    config=agents_config["tech_lead"],
    # set verbose
    verbose=True,
)

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🚦 第 5 步：Guardrails

</div>

| 函数 | 检查什么 | 挂在哪个 task |
|---|---|---|
| `security_review_output_guardrail` | 每个 `risk_level` ∈ {low, medium, high}，且 `highest_risk` 真是明细里最高的 | `review_security` |
| `review_decision_guardrail` | 结论里出现 `approve` / `request changes` / `escalate` 之一 | `make_review_decision` |

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">① 通过</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #60a5fa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">agent 交出 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">TaskOutput</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">guardrail 返回 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">(True, x)</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">x</code> 是 str → 替换 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">raw</code> 并重新解析；是 TaskOutput → 整个替换；其他 → 输出不变</span></div>

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">② 不通过</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #f87171; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">guardrail 返回 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">(False, msg)</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">msg</code> + 上次输出喂回同一个 agent 重做</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">再校验……</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #f87171; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">重试超过 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">guardrail_max_retries</code>（默认 3）→ 抛异常</span></div>

### 练习 1：`security_review_output_guardrail`
先拿到 JSON（兼容单元测试直接传的 `dict`），再查两件事：每条漏洞的等级合法；`highest_risk` 合法且等于明细里的最高等级（有 high 必须是 high，否则有 medium 必须是 medium……）。

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**为什么要 `json.loads(output.raw)` 兜底？** 源码 `task.py` 里只有 `elif not self._guardrails and not self._guardrail:` 这一分支才会在 guardrail 之前把 `raw` 解析成 `json_dict`；task 一旦挂了 guardrail，第一次调用时 `output.json_dict` 是 `None`。这是框架行为，不是代码写错。

</div>

In [8]:
# GRADED CELL: Exercise 1

import json

def security_review_output_guardrail(output):
    
    # get the (JSON) output from the TaskOutput object
    try: 
        if type(output) == dict:   # 单元测试直接传 dict
            json_output = output
        elif output.json_dict is not None:
            json_output = output.json_dict
        else:
            # when a Task has guardrails, CrewAI doesn't parse `raw` into
            # `json_dict` before the first guardrail call, so fall back to
            # parsing the raw LLM text ourselves.
            json_output = json.loads(output.raw)
    except Exception as e:
        return (False, ("Error retrieving the `json_dict` argument: "
                        f"\n{str(e)}\n"
                        "Make sure you set the output_json parameter in the Task."
                        )
                )

    # define risk levels
    valid_risk_levels = ['low', 'medium', 'high']   # 统一转小写再比，High / HIGH 也算合法
    
    # Check if security_vulnerabilities key exists
    if 'security_vulnerabilities' not in json_output:
        return (False, f"Missing 'security_vulnerabilities' key in output. Got keys: {list(json_output.keys())}")
    
    # validate that each of the risk levels has a valid value
    for vuln in json_output['security_vulnerabilities']:
        # validate the risk level
        if vuln['risk_level'].lower() not in valid_risk_levels: 
            error_message = f"Invalid risk level: {vuln['risk_level']}"
            return (False, error_message) 
    
    ### START CODE HERE ###
    # validate that the highest risk level matches the highest risk level in the vulnerabilities
    
    # if the highest risk level is not valid risk level, return an error message
    if json_output["highest_risk"].lower() not in valid_risk_levels:
        error_message = f"Invalid highest_risk level: {json_output['highest_risk']}"
        return (False, error_message)
    
    # if it is one of the valid risk levels, then check if it matches the highest 
    # risk level in the vulnerabilities
    else:
        # get all risk_level values
        risk_levels = [vuln['risk_level'].lower() for vuln in json_output['security_vulnerabilities']]    # 收集所有明细的等级
        
        # if "high" in risk levels, then highest risk level should be high
        if "high" in risk_levels: 
            if json_output["highest_risk"].lower() != "high":
                error_message = "Highest risk level does not match the highest risk level in the vulnerabilities." 
                return (False, error_message)
            
        # if high is not present and medium is in risk levels, then highest risk level should be medium
        elif "medium" in risk_levels: 
            if json_output["highest_risk"].lower() != "medium":
                error_message = "Highest risk level does not match the highest risk level in the vulnerabilities." 
                return (False, error_message)
            
        # if high and medium are not present, then lowest risk level should be low
        elif "low" in risk_levels: 
            if json_output["highest_risk"].lower() != "low":
                error_message = "Highest risk level does not match the highest risk level in the vulnerabilities." 
                return (False, error_message)
    
    ### END CODE HERE ###

    return (True, output.json_dict if type(output) != dict and output.json_dict is not None else json_output)   # 通过：返回校验后的 JSON（dict）

用一个「明细有 high、却说最高是 medium」的错误输入试一下：

In [9]:
# create the output json for testing
invalid_json = {"highest_risk": "medium", 
                        "security_vulnerabilities": [{"risk_level": "high"}, 
                                                     {"risk_level": "medium"}]}
# test the guardrail
security_review_output_guardrail(invalid_json)

(False,
 'Highest risk level does not match the highest risk level in the vulnerabilities.')

预期：`(False, 'Highest risk level does not match the highest risk level in the vulnerabilities.')`

In [10]:
# test the guardrail
unittests.test_security_review_output_guardrail(security_review_output_guardrail)

 All tests passed!



### 练习 2：`review_decision_guardrail`
取 `raw` 文本（测试时直接传 `str`），小写后检查是否包含任一决策关键词；不含就返回 `(False, 错误信息)`，让 tech lead 重写。

In [11]:
# GRADED CELL: Exercise 2

def review_decision_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw   # str（测试）或 TaskOutput（真实运行）都能处理
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        "Make sure you set the raw parameter in the Task."
                        )
                )

    # define the keywords to check for in the output
    keywords = ["approve", "request changes", "escalate"]
    
    ### START CODE HERE ###

    # check if any of the keywords are present in the output
    if not any(keyword in output.lower() for keyword in keywords):   # 子串匹配：'approved' 也算命中 approve
        # write a suitable message to the console
        error_message = "Output does not include one of the valid actionable decisions."
        return (False, error_message)

    # if all checks pass, return True, and the output
    return (True, output)   # 返回 str → CrewAI 用它替换 TaskOutput.raw

    ### END CODE HERE ###

`Elevate to human` 不是三个关键词之一（应该是 escalate），应被拒：

In [12]:
# create the raw output for testing
input = 'Final decision: Elevate to human'   # 注意：这行覆盖了内置函数 input()，本 notebook 后面没再用到它

# test the guardrail
review_decision_guardrail(input)

(False, 'Output does not include one of the valid actionable decisions.')

预期（错误信息文字可以不同）：`(False, 'Output does not include one of the valid actionable decisions.')`

In [13]:
# test the guardrail
unittests.test_review_decision_guardrail(review_decision_guardrail)

 All tests passed!



<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📋 第 6 步：Tasks

</div>

三个 task 的 `description` / `expected_output` / `name` 都来自 YAML，你只补：输出格式、guardrail、agent、context。

<div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(260px,1fr)); gap:10px; width:97%;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047;">output_json=Model</div><ul style="margin:6px 0 0 0; padding-left:18px; font-size:0.92em; line-height:1.6;"><li style="margin:2px 0;">结果存 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">TaskOutput.json_dict</code>（普通 dict）</li><li style="margin:2px 0;">本作业用它：guardrail 按 key 取值</li><li style="margin:2px 0;">Model 只用来描述 schema</li></ul></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #a78bfa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047;">output_pydantic=Model</div><ul style="margin:6px 0 0 0; padding-left:18px; font-size:0.92em; line-height:1.6;"><li style="margin:2px 0;">结果存 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">TaskOutput.pydantic</code>（模型实例）</li><li style="margin:2px 0;">用 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">.highest_risk</code> 属性访问，带类型校验</li><li style="margin:2px 0;">两者二选一，不能同时设</li></ul></div></div>

### 练习 3：Analyze Code Quality —— 质量问题分成「必须修」和「建议改」

In [14]:
# GRADED CELL: Exercise 3

# Define the pydantic model for the code quality analysis output
class CodeQualityJSON(BaseModel):
    critical_issues: list[str]   # 必须修复的问题
    minor_issues: list[str]   # 建议性改进
    reasoning: str

### START CODE HERE ###

# Create the quality analysis task
analyze_code_quality = Task(
    # Load the expected output, and name from the YAML configuration
    config=tasks_config['analyze_code_quality'],    # description / expected_output / name 来自 YAML
    # Define the output type as a pydantic model
    output_json=CodeQualityJSON,
    # Define the agent that will perform this task
    agent=senior_developer, 
)

### END CODE HERE ###

In [15]:
# test the analyze_code_quality task
unittests.test_analyze_code_quality(analyze_code_quality)

 All tests passed!



### 练习 4：Review Security —— 嵌套 schema + guardrail
`security_vulnerabilities` 是 `list[SecurityVulnerability]`：Pydantic 模型可以嵌套，生成的 JSON schema 也是嵌套的。

In [16]:
# GRADED CELL: Exercise 4

### START CODE HERE ###

# Define the pydantic model for the security vulnerabilities
class SecurityVulnerability(BaseModel):
    description: str
    risk_level: str   # 只声明 str；合法取值由 guardrail 保证（也可用 Literal['low','medium','high'] 在 schema 层限制）
    evidence: str

# Define the pydantic model for the security review output
class ReviewSecurityJSON(BaseModel):
    security_vulnerabilities: list[SecurityVulnerability] 
    blocking: bool   # 安全问题是否阻止合并
    highest_risk: str
    security_recommendations: list[str]

# Create the security review task
review_security = Task(
    # Load the expected output, agent, and name from the YAML configuration
    config=tasks_config['review_security'], 
    # Define the output type as a pydantic model
    output_json=ReviewSecurityJSON,
    # Add the security_review_output_guardrail guardrail
    guardrails=[security_review_output_guardrail],   # 列表：可挂多个，按顺序全部通过才算过
    # Define the agent that will perform this task
    agent=security_engineer 
)

### END CODE HERE ###

In [17]:
# test the review_security task
unittests.test_review_security(review_security, SecurityVulnerability, ReviewSecurityJSON)

 All tests passed!



### 练习 5：Review Decision —— Markdown 报告 + 显式 context
`context` 指定这个 task 只看哪些前置 task 的输出（不设则默认看前面所有 task）；这里必须传对象，原因见第 3 步。

In [18]:
# GRADED CELL: Exercise 5

### START CODE HERE ###

# Create the review decision task
make_review_decision = Task(
    # Load the expected output, agent, and name from the YAML configuration
    config=tasks_config['make_review_decision'], 
    # Set the Markdown parameter to get the final report in Markdown format
    markdown=True,   # 在 prompt 里要求 agent 用 Markdown 格式输出
    # Add the review_decision_guardrail guardrail
    guardrails=[review_decision_guardrail],
    # Set the context with the previous task objects
    context=[analyze_code_quality, review_security],    # 覆盖 YAML 里那份字符串 context
    # Define the agent that will perform this task
    agent=tech_lead 
)

### END CODE HERE ###

In [19]:
# test the make_review_decision task
unittests.test_make_review_decision(make_review_decision)

 All tests passed!



<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🪝 第 7 步：Before-kickoff hook 读 PR 文件

</div>

读文件不需要「智能决策」：交给 agent 用工具读又慢又费 token 还可能出错；hook 是普通 Python，确定、便宜。
CrewAI 在 kickoff 开头依次调用 `before_kickoff_callbacks`，**用返回值替换 inputs**，再拿 inputs 去填 YAML 里的 `{file_content}`。所以 hook 必须 `return inputs`。

In [20]:
# GRADED CELL: Exercise 6

# Define the execution hook to read the PR file
def read_file_hook(inputs):
    
    ### START CODE HERE ###
    
    # get the file_path from the inputs
    filename = inputs.get("file_path")   # .get：缺 key 时返回 None，交给下面的检查报错
    
    ### END CODE HERE ###

    # if the filename is not provided, raise an error
    if not filename:
        raise ValueError("Missing 'file_path' in inputs")

    # try reading the file
    try:
        with open(filename, "r") as f:
            file_contents = f.read()
    # if there are any issues, raise an error
    except Exception as e:
        raise RuntimeError(f"Failed to read file {filename}: {e}")

    
    ### START CODE HERE ###

    # add the file contents to the inputs
    inputs["file_content"] = file_contents   # key 名必须和 tasks.yaml 里的 {file_content} 一致
    
    # return the modified inputs
    return inputs   # 不 return → 后续拿到 None，占位符填不上

    ### END CODE HERE ###

In [21]:
# test the read_file_hook
unittests.test_read_file_hook(read_file_hook)

 All tests passed!



要审查的 PR（`files/code_changes.txt`，一个登录函数的 diff）——注意里面的 f-string 拼 SQL：

In [22]:
print(Path("files/code_changes.txt").read_text())

diff --git a/app/user_auth.py b/app/user_auth.py
index 8f23c4d..b9e7f2a 100644
--- a/app/user_auth.py
+++ b/app/user_auth.py
@@ -1,7 +1,32 @@
+from datetime import datetime
+import time
+
 def authenticate_user(username, password):
+    # Check if username or password is empty
+    if not username or not password:
+        return False
+    
+    # Query the database for the user
     user = db.query(f"SELECT * FROM users WHERE username = '{username}'")
+    
+    # Verify the user exists and password matches
     if user and user.password == password:
+        # Set session variables
         session['user_id'] = user.id
+        session['login_time'] = datetime.now()
+        
+        # Update last login timestamp
+        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")
+        
+        print(f"User {username} logged in successfully")
         return True
-    return False
+    else:
+        # Sleep to prevent timing attacks
+        time.sleep(1)
+       

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧑‍🤝‍🧑 第 8 步：组装 Crew（练习 7）

</div>

`memory=True` 在 crewai 1.x 是**统一 Memory**：
- 每个 agent 多两个工具 `Search memory`（召回，日志里显示为 `search_memory`）/ `Save to memory`（存），由 agent 自己决定何时用；保存时会让 LLM 提炼要点和分类，用 OpenAI embedding 做向量检索；
- 存储：LanceDB，位于 `~/Library/Application Support/<当前目录名>/memory`（目录名可用 `CREWAI_STORAGE_DIR` 改）。所以**重跑 notebook 也会召回上次的记忆**——上一次运行的日志里就能看到 `Found memories: (score=0.70) … SQL injection`。

In [23]:
# GRADED CELL: Exercise 7

### START CODE HERE ###

# Create the code review crew
crew = Crew(
    # add the list of agents
    agents=[senior_developer, security_engineer, tech_lead],
    # add the list of tasks (in order of execution) 
    tasks=[analyze_code_quality, review_security, make_review_decision],   # 默认 Process.sequential：按列表顺序执行
    # add memory to the crew
    memory=True,   # 长期记忆：跨 kickoff、跨进程保留
    # add the before-kickoff hook to read the PR file
    # you need to pass a list with the hook function from Ex 8
    before_kickoff_callbacks= [read_file_hook]   # 函数对象本身，不要加括号调用
)

### END CODE HERE ###

In [24]:
# test the crew
unittests.test_crew(crew)

 All tests passed!



<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🚀 第 9 步：Kickoff（练习 8）

</div>

只传 `file_path`，`file_content` 由 hook 补上。notebook 里用 `await crew.kickoff_async(...)`；结果用 `dill` 存成 `result.dill` 给 grader。日志里留意 guardrail 拦截时的黄色 `Guardrail … blocked … retrying` 行。

In [25]:
# GRADED CELL: Exercise 8

# define the file path for the PR file
file_path = "files/code_changes.txt" 

### START CODE HERE ###

# kickoff the crew
result = await crew.kickoff_async(
    # add the inputs dictionary with the file path
    inputs={"file_path": file_path}   # hook 会把它变成 {'file_path': ..., 'file_content': ...}
    )

### END CODE HERE ###

# Save the result to a dill file
with open('result.dill', 'wb') as f:
    dill.dump(result, f)   # 序列化整个 CrewOutput

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Developer                                                                                        │
│                                                                                                                 │
│  Task: 1. Review the code changes in the pull request. 2. Identify any style issues, potential bugs, or         │
│  maintainability concerns. 3. For each issue, decide if it's critical (must be fixed) or minor (suggestion).    │
│  PR content:  diff --git a/app/user_auth.py b/app/user_auth.py                                                  │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                            

Tool search_memory executed with result: Found memories:
- (score=0.78) The application has security vulnerabilities including a high risk of SQL injection due to string interpolation in SQL queries.
  categories: risk, assessment, security,...
Tool search_memory executed with result: Found memories:
- (score=0.70) The code contained critical issues such as SQL injection vulnerability from string interpolation in SQL queries.
  categories: security, SQL Injection Prevention, vulner...


[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Developer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "critical_issues": [                                                                                         │
│      "The code uses string interpolation in SQL queries, which exposes the application to SQL injection         │
│  vulnerabilities. This vulnerability is critical and must be fixed before approval.",                           │
│      "The implementation contains a critical security vulnerability due to the use of string interpolation in   │
│  SQL queries, which can lead to SQL injection vulnerabilities.",                                                │
│      "There is a need to replace string interpolation in SQL queries with parameterized queries to eliminate    │
│  the risk of SQL injection."                                                                                    │
│    ],                                                                                                           │
│    "minor_issues": [                                                                                            │
│      "Removing time.sleep(1) from the login function may enhance system responsiveness and should be            │
│  considered.",                                                                                                  │
│      "The logout_user function lacks error handling, which could improve maintainability and robustness.",      │
│      "Replacing print statements with a logging mechanism can enhance error tracking and monitoring.",          │
│      "Enhancing feedback during the logout process can improve user experience."                                │
│    ],                                                                                                           │
│    "reasoning": "The identified critical issues relate to security vulnerabilities, specifically SQL injection  │
│  risks due to the use of string interpolation in SQL queries. These issues pose significant risks and must be   │
│  addressed prior to approval of the pull request. The minor issues are suggestions that, while not blocking     │
│  approval, could enhance code maintainability, user experience, and overall quality."                           │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Task: 1. Review the findings from both the code quality and security analyses. 2. Determine if the pull        │
│  request can be automatically approved based on these findings. 3. If approval is not possible, decide whether  │
│  to request specific changes from the developer or escalate to human review. 4. Provide a clear explanation of  │
│  your decision and any required next steps. PR content:  diff --git a/app/user_auth.py b/app/user_auth.py       │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                     

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "security_vulnerabilities": [                                                                                │
│      {                                                                                                          │
│        "description": "The code uses string interpolation in SQL queries, which exposes the application to SQL  │
│  injection vulnerabilities.",                                                                                   │
│        "risk_level": "high",                                                                                    │
│        "evidence": "The implementation uses string interpolation in SQL queries."                               │
│      },                                                                                                         │
│      {                                                                                                          │
│        "description": "There is a critical security vulnerability due to the use of string interpolation in     │
│  SQL queries.",                                                                                                 │
│        "risk_level": "high",                                                                                    │
│        "evidence": "The identified critical issues relate to security vulnerabilities, specifically SQL         │
│  injection risks."                                                                                              │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "blocking": true,                                                                                            │
│    "highest_risk": "high",                                                                                      │
│    "security_recommendations": [                                                                                │
│      "Replace string interpolation in SQL queries with parameterized queries to eliminate the risk of SQL       │
│  injection.",                                                                                                   │
│      "Add error handling in the logout_user function to improve robustness.",                                   │
│      "Consider removing time.sleep(1) from the login function to enhance system responsiveness.",               │
│      "Replace print statements with a logging mechanism to improve error tracking and monitoring.",             │
│      "Enhance feedback during the logout process to improve user experience."                                   │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Lead                                                                                               │
│                                                                                                                 │
│  Task: 1. Based on the results from 'analyze_code_quality' and 'review_security', make a final review           │
│  decision. 2. Decide whether to approve the changes, request changes, or escalate to human review. 3.           │
│  Summarize the key findings from both analyses to support your decision. 4. If requesting changes, provide a    │
│  clear list of what must be fixed. 5. If approving, include appropriate comments to accompany the approval. 6.  │
│  If escalating, explain why human review is necessary. 7. Include any additional recommendations or             │
│  observations. PR content:  diff --git a/app/user_auth.py b/app/user_auth.py                                    │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfull

Tool search_memory executed with result: Found memories:
- (score=0.69) The analysis found a high risk level for the SQL injection vulnerability, and this issue should block approval of the pull request.
  categories: risk, assessment, Secur...
Tool search_memory executed with result: Found memories:
- (score=0.73) The identified critical issues, particularly the SQL injection vulnerability, pose significant security risks and must be addressed prior to approval.
  categories: secu...


[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Lead                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Final Review Decision                                                                                        │
│                                                                                                                 │
│  Based on the evaluations from the code quality and security analyses, the final decision is to **request       │
│  changes** before this pull request (PR) can be approved.                                                       │
│                                                                                                                 │
│  ## Summary of Key Findings                                                                                     │
│                                                                                                                 │
│  1. **Critical Issues**:                                                                                        │
│     - The code uses **string interpolation** in SQL queries, which exposes the application to **SQL injection   │
│  vulnerabilities**. This is classified as a high-risk security vulnerability that needs to be addressed         │
│  immediately.                                                                                                   │
│     - It is necessary to replace string interpolation with **parameterized queries** to eliminate the risk of   │
│  SQL injection.                                                                                                 │
│                                                                                                                 │
│  2. **Minor Issues**:                                                                                           │
│     - The `logout_user` function lacks proper **error handling**, which would enhance maintainability.          │
│     - The use of `time.sleep(1)` in the login function may reduce system responsiveness, and its removal        │
│  should be considered.                                                                                          │
│     - All `print` statements should be replaced with a **logging mechanism** to improve error tracking.         │
│     - Enhancements to user feedback during the logout process could further improve the user experience.        │
│                                                                                                                 │
│  ## List of Required Fixes                                                                                      │
│                                                                                                                 │
│  1. **Security Fixes**:                                                                                         │
│     - Replace string interpolation in SQL queries with **parameterized queries** to prevent SQL injection.      │
│                                                                                                                 │
│  2. **Functionality Improvements**:                                                                             │
│     - Improve error handling in the `logout_user` function for better feedback.                                 │
│     - Consider removing `time.sleep(1)` from the `authenticate_user` function for better responsiveness.        │
│     - Replace all `print` statements with appropriate l

看每个 task 的结构化输出（`crew.kickoff` 后每个 Task 对象的 `.output` 都有值）：

In [26]:
for t in crew.tasks:
    out = t.output
    print(f"[{out.name}] json_dict={'有' if out.json_dict else 'None'}  raw 前 150 字：{out.raw[:150]!r}")

[Analyze Code Quality] json_dict=有  raw 前 150 字：'{\n  "critical_issues": [\n    "The code uses string interpolation in SQL queries, which exposes the application to SQL injection vulnerabilities. This '
[Review Security] json_dict=None  raw 前 150 字：'{\n  "security_vulnerabilities": [\n    {\n      "description": "The code uses string interpolation in SQL queries, which exposes the application to SQL '
[Review Decision] json_dict=None  raw 前 150 字：'# Final Review Decision\n\nBased on the evaluations from the code quality and security analyses, the final decision is to **request changes** before thi'


<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**为什么 `Review Security` 的 `json_dict` 是 `None`？** 它挂了 guardrail，所以 CrewAI 先不解析 JSON（见练习 1）；guardrail 通过后，源码只在返回值是 **str**（替换 `raw` 并重新解析）或 **TaskOutput**（整个替换）时才更新输出。练习 1 返回的是 **dict**，于是输出保持原样，`json_dict` 一直是 `None`，JSON 只在 `raw` 里。不影响结果：tech lead 通过 context 读的就是 `raw` 文本。若想拿到 `json_dict`，guardrail 可返回 `(True, output.raw)`。

</div>

最终报告：

In [27]:
# clean the markdown output, in case the report is wrapped with code fences
clean_report = clean_markdown(result.raw)

# display the final report as markdown
display(Markdown(clean_report))

# Final Review Decision

Based on the evaluations from the code quality and security analyses, the final decision is to **request changes** before this pull request (PR) can be approved.

## Summary of Key Findings

1. **Critical Issues**:
   - The code uses **string interpolation** in SQL queries, which exposes the application to **SQL injection vulnerabilities**. This is classified as a high-risk security vulnerability that needs to be addressed immediately. 
   - It is necessary to replace string interpolation with **parameterized queries** to eliminate the risk of SQL injection.

2. **Minor Issues**:
   - The `logout_user` function lacks proper **error handling**, which would enhance maintainability.
   - The use of `time.sleep(1)` in the login function may reduce system responsiveness, and its removal should be considered.
   - All `print` statements should be replaced with a **logging mechanism** to improve error tracking.
   - Enhancements to user feedback during the logout process could further improve the user experience.

## List of Required Fixes

1. **Security Fixes**:
   - Replace string interpolation in SQL queries with **parameterized queries** to prevent SQL injection.

2. **Functionality Improvements**:
   - Improve error handling in the `logout_user` function for better feedback.
   - Consider removing `time.sleep(1)` from the `authenticate_user` function for better responsiveness.
   - Replace all `print` statements with appropriate logging calls.
   - Enhance user feedback during the logout process.

## Comments for Approval

If the necessary changes are made, include comments that emphasize the importance of security and the need for proper coding practices. For example:

- "Security vulnerabilities have been addressed, and best practices for coding have been implemented. Great job on adopting feedback and improving code quality!"

## Additional Recommendations

- Conduct thorough testing after making the proposed changes to ensure that all issues have been resolved effectively.
- Regularly review database access patterns for security and performance, and establish a coding standard for error handling and logging across the project.
- Update documentation to reflect the changes made regarding security practices and logging mechanisms.

---

By addressing these critical and minor issues, we can enhance the security and quality of the application, ensuring a more robust solution moving forward.

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 坑**

| 坑 | 现象 | 原因 / 对策 |
|---|---|---|
| `base_url=os.getenv(...)` 本地返回 `None` | `SerperDevTool` 报 pydantic `ValidationError` | `base_url` 是严格 `str`；`os.getenv(...) or "https://google.serper.dev"` |
| guardrail 里直接用 `output.json_dict` | `'NoneType' object has no attribute ...` | 挂了 guardrail 的 task 不会提前解析 JSON；兜底 `json.loads(output.raw)` |
| YAML 里的 `context` 字符串列表 | `AttributeError: 'str' object has no attribute 'get'` | 非 `@CrewBase` 场景显式传 `context=[task对象]` |
| hook 忘了 `return inputs` | inputs 变 `None`，`{file_content}` 填不上 | before-kickoff hook 的返回值会替换 inputs |
| 以为 `search_url` 能限定站点 | 照样全网搜索 | `SerperDevTool` 没这个字段，被静默忽略 |
| 遇到 `None` 报错就加 `if x is None: skip` | 掩盖真正原因 | 先查这个值本该从哪来、为什么没来——常常要读框架源码 |

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**🔁 复习自测**（先自己答，再点开看答案）

<details><summary><b>1. guardrail 的返回值约定是什么？失败后发生什么？</b></summary>

返回 `(bool, x)`。`True` 时 `x` 是结果（str 会替换 `raw` 并重新解析）；`False` 时 `x` 是错误信息，CrewAI 把它连同上次输出喂回同一个 agent 重做，最多 `guardrail_max_retries`（默认 3）次，仍失败抛异常。

</details>

<details><summary><b>2. 为什么练习 1 要写 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">json.loads(output.raw)</code> 兜底？</b></summary>

task 设了 guardrail 时，CrewAI 不会在第一次调用 guardrail 前把 `raw` 解析进 `json_dict`（`task.py` 的 `elif not self._guardrails and not self._guardrail:` 分支），所以 `json_dict` 是 `None`。

</details>

<details><summary><b>3. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">read_file_hook</code> 为什么用 hook 而不是给 agent 一个读文件工具？</b></summary>

读文件是确定性 I/O，不需要推理；hook 用纯 Python 做，更快、更便宜、不会出错。它在 agent 开始前运行，返回值替换 inputs，从而填进 YAML 的 `{file_content}`。

</details>

<details><summary><b>4. YAML 里写了 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">context</code>，为什么代码里还要传 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">context=[...]</code>？</b></summary>

YAML 里是字符串，只有 `@CrewBase` 会把它解析成 task 对象；直接 `Task(config=...)` 会报错。代码里显式传的参数优先于 config。

</details>

<details><summary><b>5. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">memory=True</code> 的记忆存在哪？下次重跑还在吗？</b></summary>

LanceDB，位于 `~/Library/Application Support/<当前目录名>/memory`。在，跨 kickoff、跨进程都在；agent 通过 `Search memory` / `Save to memory` 工具读写。

</details>

<details><summary><b>6. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">memory</code> 和 task 的 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">context</code> 有什么区别？</b></summary>

`context` 是**同一次 kickoff 内**把前置 task 的输出传给后面的 task；`memory` 是**跨多次 kickoff** 的长期记忆。

</details>

<details><summary><b>7. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">output_json</code> 和 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">output_pydantic</code> 怎么选？</b></summary>

`output_json` → `json_dict`（dict，按 key 取）；`output_pydantic` → `pydantic`（模型实例，按属性取、带类型校验）。二选一。

</details>

<details><summary><b>8. M1 学的 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">after_kickoff_callbacks</code> 能做什么？</b></summary>

全部 task 跑完后调用，收到最终结果，例如 `save_file_hook` 把报告存成本地 Markdown 文件。

</details>

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：`output_json` 让输出可被程序解析，`guardrails` 自动校验并重试，`before_kickoff_callbacks` 把确定性 I/O 移出 agent，`memory` 让 crew 跨运行保持一致——这四件事就是多 agent 系统从 demo 走向 production 的标准路径。

</div>